In [3]:
import json

from src.server.models.user_constants import uc
from src.server.models.user_quota_model import QuotaPortalInternalBatchContext, QuotaPortalUserContext
from src.server.models.gpt_tagger.prompt_handlers_v2 import PromptHandlerPriorityTags
from src.db.mysql import DbConnection

conn = DbConnection()

GENRES = 21
TOPICS = 43

rows = conn.select_all("""
SELECT t.app_id_int
  FROM tagged_data.tagging_state t,
       scraped_data.apps a
 WHERE t.state = 'tagged'
   AND a.app_id_int = t.app_id_int
   AND a.released IS NOT null
   AND a.installs > 1000
   AND NOT EXISTS (
       SELECT 1
         FROM tagged_data.tags tt,
              app.def_sheet_platform_product p
        WHERE tt.app_id_int = t.app_id_int
          AND tt.tag_rank != 0
          AND tt.tag_id_int = p.tag_id_int
          AND p.subcategory_int = %s
	)
	AND t.app_id_int NOT IN (
	   SELECT u.app_id_int FROM tagged_data.gpt_unknown_ranked_tags u
	)
UNION
SELECT t.app_id_int
  FROM tagged_data.tagging_state t,
       scraped_data.apps a
 WHERE t.state = 'tagged'
   AND a.app_id_int = t.app_id_int
   AND a.released IS NOT null
   AND a.installs > 1000
   AND NOT EXISTS (
       SELECT 1
         FROM tagged_data.tags tt,
              app.def_sheet_platform_product p
        WHERE tt.app_id_int = t.app_id_int
          AND tt.tag_rank != 0
          AND tt.tag_id_int = p.tag_id_int
          AND p.subcategory_int = %s
	)
	AND t.app_id_int NOT IN (
	   SELECT u.app_id_int FROM tagged_data.gpt_unknown_ranked_tags u
	)
GROUP BY app_id_int
ORDER BY app_id_int
""", [GENRES, TOPICS])

#rows = [{"app_id_int": 288}]
quota_context = QuotaPortalInternalBatchContext(QuotaPortalUserContext.QUOTA_OPENAI, "tag app")

for row in rows:
    app_id_int = row["app_id_int"]

    print(f"App id: {app_id_int}")

    rows_tags = conn.select_all("""
    SELECT t.tag_id_int
      FROM tagged_data.tags t,
           app.def_sheet_platform_product p
     WHERE t.app_id_int = %s
       AND p.tag_id_int = t.tag_id_int
       AND p.subcategory_int IN (%s, %s)
    """, [app_id_int, GENRES, TOPICS])

    tags_ids_int = []
    for row_tag in rows_tags:
        tags_ids_int.append(row_tag["tag_id_int"])

    conn.query("""
    DELETE FROM tagged_data.tagger_requests r
     WHERE r.app_id_int = %s
       AND r.prompt_context = 'app'
    """, [app_id_int])
    request_id = conn.insert("""
    INSERT INTO tagged_data.tagger_requests
    (app_id_int, state, prompt_context)
    VALUES
    (%s, 'queue', 'prio rank fix')
    """, [app_id_int])
    conn.commit()

    prompt_handler = PromptHandlerPriorityTags(
        conn=conn,
        user_id=uc.get_system_batch_user_id(),
        quota_context=quota_context,
        app_id_int=app_id_int,
        tags_ids_int=tags_ids_int,
        retry_prompt_row_id=0
    )
    res = prompt_handler.run()
    if res["state"] == "done":
        parsed_result = prompt_handler.parse_result(save_unknown=True)
        if parsed_result["state"] == "done":
            tags_details_ranked = parsed_result["tags_details"]
            bulk_data = []
            for tag_detail in parsed_result["tags_details"]:
                bulk_data.append((tag_detail["tag_rank"], tag_detail["tag_id_int"], app_id_int))
            if len(bulk_data) > 0:
                conn.query("""
                UPDATE tagged_data.tags t
                  inner join app.def_sheet_platform_product p
                  ON p.tag_id_int = t.tag_id_int
                  AND p.subcategory_int IN (%s, %s)
                 SET t.tag_rank = 0
                 WHERE t.app_id_int = %s
                """, [GENRES, TOPICS, app_id_int])
                conn.bulk("""
                UPDATE tagged_data.tags t
                   SET t.tag_rank = %s
                 WHERE t.tag_id_int = %s
                   AND t.app_id_int = %s
                """, bulk_data)

            conn.query("""
            DELETE FROM tagged_data.tagger_requests r
             WHERE r.id = %s
            """, [request_id])

            conn.commit()
            print(f"App id: {app_id_int} DONE")
        else:
            conn.query("""
            UPDATE tagged_data.tagger_requests r
               SET r.progress_data = %s,
                   r.t = NOW(),
                   r.state = 'retry'
             WHERE r.id = %s
            """, [json.dumps(parsed_result), request_id])
            conn.commit()
            print(f"App id: {app_id_int} ERROR")
    else:
        conn.query("""
        UPDATE tagged_data.tagger_requests r
           SET r.progress_data = %s,
               r.t = NOW(),
               r.state = 'retry'
         WHERE r.id = %s
        """, [json.dumps(res), request_id])
        conn.commit()
        print(f"App id: {app_id_int} ERROR")

conn.commit()
conn.close()

App id: 288
App id: 288 DONE


In [2]:
conn.close()